# Diamonds Price Prediction Analysis

## Table of Contents
1. Introduction
2. Data Loading and Preprocessing
3. Data Splitting (Train, Validation, Test)
4. Feature Scaling
5. Model Training and Evaluation
   - Linear Regression
   - Random Forest
   - Decision Tree
   - K-Nearest Neighbors (KNN)
6. Model Performance Comparison
7. Conclusion

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/diamonds/diamonds.csv


## 1. Introduction
Diamond price prediction is a regression problem where we aim to estimate the price of a diamond based on various attributes such as cut, color, clarity, and dimensions. We use multiple machine learning models and evaluate their performance.


In [2]:
from sklearn.preprocessing import StandardScaler  # Feature scaling
import matplotlib.pyplot as plt  # Visualization
import numpy as np  # Numerical operations
import os  # File handling
import pandas as pd  # Data handling

In [3]:
# Download dataset from Kaggle
import kagglehub
path = kagglehub.dataset_download("shivam2503/diamonds")
print("Path to dataset files:", path)


Path to dataset files: /kaggle/input/diamonds


In [4]:
# List downloaded files
files = os.listdir(path)
print(files)  # Should display 'diamonds.csv'

['diamonds.csv']


## 2. Data Loading and Preprocessing

In [5]:
# Load the dataset
df = pd.read_csv(os.path.join(path, 'diamonds.csv'))
print(df.head())  # Display first 5 rows

   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2           3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07   
3           4   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23   
4           5   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35   

      z  
0  2.43  
1  2.31  
2  2.31  
3  2.63  
4  2.75  


In [6]:
# Display dataset information (column names, data types, missing values)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53940 non-null  int64  
 1   carat       53940 non-null  float64
 2   cut         53940 non-null  object 
 3   color       53940 non-null  object 
 4   clarity     53940 non-null  object 
 5   depth       53940 non-null  float64
 6   table       53940 non-null  float64
 7   price       53940 non-null  int64  
 8   x           53940 non-null  float64
 9   y           53940 non-null  float64
 10  z           53940 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB


In [7]:
# Summary statistics of numerical features
df.describe()

,Unnamed: 0,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,26970.500000,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,15571.281097,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,1.000000,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,13485.750000,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,26970.500000,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,40455.250000,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,53940.000000,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


In [8]:
# Check for missing values
print(df.isnull().sum())

Unnamed: 0    0
carat         0
cut           0
color         0
clarity       0
depth         0
table         0
price         0
x             0
y             0
z             0
dtype: int64


In [9]:
# Drop unnecessary columns (Assuming 'Unnamed: 0' exists and is an index column)
df = df.drop(columns=['Unnamed: 0','table', 'depth'], errors='ignore')

In [10]:
# Encode categorical features using one-hot encoding
df = pd.get_dummies(df, columns=['cut', 'color', 'clarity'], drop_first=True)

## 3. Data Splitting (Train, Validation, Test)

In [11]:
# Define features and target variable
X = df.drop(columns=['price'])  # Features
y = df['price']  # Target variable

In [12]:
# Split data into training (70%), validation (15%), and test (15%) sets
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

## 4. Feature Scaling

In [13]:
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


## 5. Model Training and Evaluation

In [14]:
### Linear Regression
from sklearn.linear_model import LinearRegression
model_lr = LinearRegression()
model_lr.fit(X_train_scaled, y_train)

LinearRegression()

In [15]:
# Evaluate the model on validation set
from sklearn.metrics import mean_squared_error, r2_score
y_val_pred = model_lr.predict(X_val_scaled)
val_r2_lr = r2_score(y_val, y_val_pred)
val_rmse_lr = np.sqrt(mean_squared_error(y_val, y_val_pred))


In [16]:
# Final evaluation on test set
y_test_pred = model_lr.predict(X_test_scaled)
test_r2_lr = r2_score(y_test, y_test_pred)
test_rmse_lr = np.sqrt(mean_squared_error(y_test, y_test_pred))

In [17]:
### Random Forest
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [18]:
# Evaluate Random Forest on validation set
y_val_pred_rf = model_rf.predict(X_val)
val_r2_rf = r2_score(y_val, y_val_pred_rf)
val_rmse_rf = np.sqrt(mean_squared_error(y_val, y_val_pred_rf))

In [19]:
# Final evaluation on test set
y_test_pred_rf = model_rf.predict(X_test)
test_r2_rf = r2_score(y_test, y_test_pred_rf)
test_rmse_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))


In [20]:
### Decision Tree
from sklearn.tree import DecisionTreeRegressor
model_dt = DecisionTreeRegressor(random_state=42)
model_dt.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [21]:
# Evaluate Decision Tree on validation set
y_val_pred_dt = model_dt.predict(X_val)
val_r2_dt = r2_score(y_val, y_val_pred_dt)
val_rmse_dt = np.sqrt(mean_squared_error(y_val, y_val_pred_dt))

In [22]:
# Final evaluation on test set
y_test_pred_dt = model_dt.predict(X_test)
test_r2_dt = r2_score(y_test, y_test_pred_dt)
test_rmse_dt = np.sqrt(mean_squared_error(y_test, y_test_pred_dt))


In [23]:
### K-Nearest Neighbors (KNN)
from sklearn.neighbors import KNeighborsRegressor
model_knn = KNeighborsRegressor(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

KNeighborsRegressor()

In [24]:
# Evaluate KNN on validation set
y_val_pred_knn = model_knn.predict(X_val_scaled)
val_r2_knn = r2_score(y_val, y_val_pred_knn)
val_rmse_knn = np.sqrt(mean_squared_error(y_val, y_val_pred_knn))

In [25]:
# Final evaluation on test set
y_test_pred_knn = model_knn.predict(X_test_scaled)
test_r2_knn = r2_score(y_test, y_test_pred_knn)
test_rmse_knn = np.sqrt(mean_squared_error(y_test, y_test_pred_knn))


## 6. Model Performance Comparison

In [26]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Decision Tree', 'KNN'],
    'Validation R2': [val_r2_lr, val_r2_rf, val_r2_dt, val_r2_knn],
    'Validation RMSE': [val_rmse_lr, val_rmse_rf, val_rmse_dt, val_rmse_knn],
    'Test R2': [test_r2_lr, test_r2_rf, test_r2_dt, test_r2_knn],
    'Test RMSE': [test_rmse_lr, test_rmse_rf, test_rmse_dt, test_rmse_knn]
})
print(results)

               Model  Validation R2  Validation RMSE   Test R2    Test RMSE
0  Linear Regression       0.922153      1090.696193  0.917658  1144.537611
1      Random Forest       0.974676       622.078350  0.975298   626.886521
2      Decision Tree       0.955665       823.110402  0.957552   821.767104
3                KNN       0.966857       711.666858  0.969075   701.415349


## 7. Conclusion
Based on R2 Score and RMSE, the model with the highest performance should be selected for final deployment.

If Random Forest has the highest R² and lowest RMSE, it is the best choice.
If Decision Tree has close performance but lower test R², it is overfitting.
If Linear Regression is performing well, the data follows a more linear trend.
If KNN is the worst, it's because it doesn’t scale well with many features.


Should You Choose Test or Validation RMSE?
The choice depends on your goal:

# Validation RMSE is used for model selection.
1.Helps tune hyperparameters.\
2.Prevents overfitting to the training set.\
3.If a model performs well on validation but poorly on the test, it might be overfitting.
# Test RMSE is used for final evaluation.
1.Gives an unbiased estimate of real-world performance.\
2.If this is high, the model might not generalize well.
# Final Decision
1.Use Validation RMSE to pick the best model.
2.Use Test RMSE to report the final model’s performance.

From the above it is the Random Forest which is the best model

Thank you everyone